# Auto Loader - Ingestion danych warsztatowych

Inkrementalne wczytywanie plików parquet z Unity Catalog Volumes do tabel Delta.

| Widget | Wartości | Opis |
|--------|----------|------|
| `TRIGGER_MODE` | `availableNow` / `continuous` | **availableNow** – przetwarza wszystkie dostępne pliki i zatrzymuje się (używaj do wstępnego ładowania i testów Auto Loader z danymi jednodniowymi). **continuous** – strumień działa ciągle, pobiera nowe pliki co 30 s. |
| `SINGLE_TABLE` | nazwa tabeli lub puste | Zostaw puste, aby wczytać wszystkie 20 tabel. Ustaw np. `dim_locations`, aby uruchomić jedną tabelę. |

**Struktura woluminów:**
```
/Volumes/fake_car_workshop_franchise/
  dim/
    dim_parquet_files/       ← źródło (output generatora)
    autoloader_checkpoints/  ← checkpointy + lokalizacje schematów (ten notebook)
  fact/
    fact_parquet_files/      ← źródło
    autoloader_checkpoints/  ← checkpointy + lokalizacje schematów
```

In [0]:
import os
from pyspark.sql.functions import col

print("Imports OK")

In [0]:
dbutils.widgets.dropdown(
    'TRIGGER_MODE', 'availableNow', ['availableNow', 'continuous'],
    label='Trigger Mode'
)
dbutils.widgets.text(
    'SINGLE_TABLE', '',
    label='Single Table  (blank = all, e.g. dim_locations)'
)

TRIGGER_MODE = dbutils.widgets.get('TRIGGER_MODE')
SINGLE_TABLE = dbutils.widgets.get('SINGLE_TABLE').strip()

print(f'TRIGGER_MODE = {TRIGGER_MODE}')
print(f'SINGLE_TABLE = {SINGLE_TABLE or "(all tables)"}')

In [0]:
CATALOG              = 'fake_car_workshop_franchise_pl'

DIM_PARQUET_BASE     = f'/Volumes/{CATALOG}/dim/dim_parquet_files'
FACT_PARQUET_BASE    = f'/Volumes/{CATALOG}/fact/fact_parquet_files'
DIM_CHECKPOINT_BASE  = f'/Volumes/{CATALOG}/dim/autoloader_checkpoints'
FACT_CHECKPOINT_BASE = f'/Volumes/{CATALOG}/fact/autoloader_checkpoints'

print(f'Source  DIM :  {DIM_PARQUET_BASE}')
print(f'Source  FACT:  {FACT_PARQUET_BASE}')
print(f'Chkpts  DIM :  {DIM_CHECKPOINT_BASE}')
print(f'Chkpts  FACT:  {FACT_CHECKPOINT_BASE}')

In [0]:
def schema_to_ddl(d):
    return ", ".join(f"`{c}` {t}" for c, t in d.items())


TABLE_SCHEMAS = {
    # ── wymiarowe ───────────────────────────────────────────────────────
    "dim_locations": {
        "location_id": "BIGINT",
        "location_code": "STRING",
        "nazwa": "STRING",
        "typ": "STRING",
        "ulica": "STRING",
        "miasto": "STRING",
        "wojewodztwo": "STRING",
        "kod_pocztowy": "STRING",
        "latitude": "DOUBLE",
        "longitude": "DOUBLE",
        "telefon": "STRING",
        "email": "STRING",
        "kierownik_id": "BIGINT",
        "liczba_stanowisk": "BIGINT",
        "powierzchnia_m2": "BIGINT",
        "data_otwarcia": "DATE",
        "czy_aktywna": "BOOLEAN",
    },
    "dim_employees": {
        "employee_id": "BIGINT",
        "employee_code": "STRING",
        "imie": "STRING",
        "nazwisko": "STRING",
        "pesel": "STRING",
        "stanowisko": "STRING",
        "location_id": "BIGINT",
        "data_zatrudnienia": "DATE",
        "data_zwolnienia": "DATE",
        "stawka_godzinowa": "DOUBLE",
        "czy_aktywny": "BOOLEAN",
    },
    "dim_customers": {
        "customer_id": "BIGINT",
        "customer_code": "STRING",
        "typ_klienta": "STRING",
        "imie": "STRING",
        "nazwisko": "STRING",
        "nazwa_firmy": "STRING",
        "nip": "STRING",
        "email": "STRING",
        "telefon": "STRING",
        "miasto": "STRING",
        "kod_pocztowy": "STRING",
        "data_rejestracji": "DATE",
        "preferowana_lokalizacja_id": "BIGINT",
        "zgoda_marketing": "BOOLEAN",
    },
    "dim_vehicles": {
        "vehicle_id": "BIGINT",
        "customer_id": "BIGINT",
        "marka": "STRING",
        "model": "STRING",
        "rocznik": "BIGINT",
        "vin": "STRING",
        "nr_rejestracyjny": "STRING",
        "typ_paliwa": "STRING",
        "pojemnosc_silnika": "DOUBLE",
        "moc_km": "BIGINT",
        "kolor": "STRING",
        "przebieg_km": "BIGINT",
        "data_pierwszej_rejestracji": "DATE",
    },
    "dim_products": {
        "product_id": "BIGINT",
        "product_code": "STRING",
        "nazwa": "STRING",
        "kategoria": "STRING",
        "producent": "STRING",
        "cena_zakupu_netto": "DOUBLE",
        "cena_sprzedazy_netto": "DOUBLE",
        "vat_procent": "BIGINT",
        "jednostka": "STRING",
        "waga_kg": "DOUBLE",
        "min_stan_magazynowy": "BIGINT",
        "czy_aktywny": "BOOLEAN",
    },
    "dim_services": {
        "service_id": "BIGINT",
        "service_code": "STRING",
        "nazwa": "STRING",
        "kategoria": "STRING",
        "cena_min_netto": "BIGINT",
        "cena_max_netto": "BIGINT",
        "szacowany_czas_min": "BIGINT",
        "czy_aktywna": "BOOLEAN",
    },
    "dim_suppliers": {
        "supplier_id": "BIGINT",
        "supplier_code": "STRING",
        "nazwa": "STRING",
        "nip": "STRING",
        "miasto": "STRING",
        "adres": "STRING",
        "kod_pocztowy": "STRING",
        "telefon": "STRING",
        "email": "STRING",
        "osoba_kontaktowa": "STRING",
        "warunki_platnosci_dni": "BIGINT",
        "min_wartosc_zamowienia": "DOUBLE",
        "czy_aktywny": "BOOLEAN",
    },
    # ── faktowe ─────────────────────────────────────────────────────────
    "fact_work_orders": {
        "work_order_id": "BIGINT",
        "work_order_code": "STRING",
        "location_id": "BIGINT",
        "customer_id": "BIGINT",
        "vehicle_id": "BIGINT",
        "mechanic_id": "BIGINT",
        "data_przyjecia": "DATE",
        "data_zakonczenia": "DATE",
        "status": "STRING",
        "przebieg_przy_przyjecia": "BIGINT",
        "uwagi_klienta": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_work_order_items": {
        "wo_item_id": "BIGINT",
        "work_order_id": "BIGINT",
        "typ_pozycji": "STRING",
        "service_id": "BIGINT",
        "product_id": "BIGINT",
        "ilosc": "BIGINT",
        "cena_jednostkowa_netto": "DOUBLE",
        "wartosc_netto": "DOUBLE",
        "vat_procent": "BIGINT",
        "wartosc_brutto": "DOUBLE",
        "rabat_procent": "BIGINT",
    },
    "fact_sales_transactions": {
        "transaction_id": "BIGINT",
        "transaction_code": "STRING",
        "location_id": "BIGINT",
        "customer_id": "BIGINT",
        "employee_id": "BIGINT",
        "data_transakcji": "TIMESTAMP",
        "metoda_platnosci": "STRING",
        "nr_paragonu": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_sales_items": {
        "sales_item_id": "BIGINT",
        "transaction_id": "BIGINT",
        "product_id": "BIGINT",
        "ilosc": "BIGINT",
        "cena_jednostkowa_netto": "DOUBLE",
        "rabat_procent": "BIGINT",
        "wartosc_netto": "DOUBLE",
        "vat_procent": "BIGINT",
        "wartosc_brutto": "DOUBLE",
    },
    "fact_invoices": {
        "invoice_id": "BIGINT",
        "invoice_code": "STRING",
        "typ_dokumentu": "STRING",
        "source_type": "STRING",
        "source_id": "BIGINT",
        "customer_id": "BIGINT",
        "location_id": "BIGINT",
        "data_wystawienia": "DATE",
        "data_sprzedazy": "DATE",
        "termin_platnosci": "DATE",
        "wartosc_netto": "DOUBLE",
        "wartosc_vat": "DOUBLE",
        "wartosc_brutto": "DOUBLE",
        "status": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_payments": {
        "payment_id": "BIGINT",
        "invoice_id": "BIGINT",
        "data_platnosci": "DATE",
        "kwota": "DOUBLE",
        "metoda_platnosci": "STRING",
        "status": "STRING",
        "numer_transakcji": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_inventory_movements": {
        "movement_id": "BIGINT",
        "product_id": "BIGINT",
        "location_id": "BIGINT",
        "typ_ruchu": "STRING",
        "ilosc": "BIGINT",
        "data_ruchu": "DATE",
        "dokument_zrodlowy": "STRING",
        "nr_dokumentu": "STRING",
        "wartosc_netto": "DOUBLE",
        "uwagi": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_appointments": {
        "appointment_id": "BIGINT",
        "customer_id": "BIGINT",
        "vehicle_id": "BIGINT",
        "location_id": "BIGINT",
        "service_id": "BIGINT",
        "data_rezerwacji": "DATE",
        "data_wizyty": "TIMESTAMP",
        "status": "STRING",
        "kanal_rezerwacji": "STRING",
        "uwagi": "STRING",
        "rok": "BIGINT",
        "miesiac": "BIGINT",
    },
    "fact_purchase_orders": {
        "po_id": "BIGINT",
        "po_code": "STRING",
        "supplier_id": "BIGINT",
        "location_id": "BIGINT",
        "data_zamowienia": "DATE",
        "data_dostawy_planowana": "DATE",
        "data_dostawy_rzeczywista": "DATE",
        "wartosc_netto": "DOUBLE",
        "wartosc_brutto": "DOUBLE",
        "status": "STRING",
        "rok": "BIGINT",
    },
    "fact_purchase_order_items": {
        "po_item_id": "BIGINT",
        "po_id": "BIGINT",
        "product_id": "BIGINT",
        "ilosc_zamowiona": "BIGINT",
        "ilosc_dostarczona": "BIGINT",
        "cena_jednostkowa_netto": "DOUBLE",
        "wartosc_netto": "DOUBLE",
    },
    "fact_customer_feedback": {
        "feedback_id": "BIGINT",
        "customer_id": "BIGINT",
        "location_id": "BIGINT",
        "work_order_id": "BIGINT",
        "data_opinii": "DATE",
        "ocena": "BIGINT",
        "komentarz": "STRING",
        "kategoria": "STRING",
        "kanal": "STRING",
    },
    "fact_loyalty_program": {
        "loyalty_id": "BIGINT",
        "customer_id": "BIGINT",
        "data_zdarzenia": "DATE",
        "typ_zdarzenia": "STRING",
        "punkty": "BIGINT",
        "opis": "STRING",
        "saldo_po": "BIGINT",
        "poziom": "STRING",
    },
    "fact_employee_schedules": {
        "schedule_id": "BIGINT",
        "employee_id": "BIGINT",
        "data": "DATE",
        "godzina_start": "BIGINT",
        "godzina_koniec": "BIGINT",
        "typ_zmiany": "STRING",
        "nadgodziny_h": "BIGINT",
        "obecnosc": "STRING",
    },
}

# tabele faktowe z kolumnami partycji w strukturze katalogów parquet
PARTITIONED_TABLES = {
    "fact_work_orders": ["rok", "miesiac"],
    "fact_sales_transactions": ["rok", "miesiac"],
    "fact_invoices": ["rok", "miesiac"],
    "fact_payments": ["rok", "miesiac"],
    "fact_inventory_movements": ["rok", "miesiac"],
    "fact_appointments": ["rok", "miesiac"],
    "fact_purchase_orders": ["rok"],
}

print(f"Schematy załadowane: {len(TABLE_SCHEMAS)} tabel")

In [0]:
# Keeps references to running streams when TRIGGER_MODE = continuous
active_streams = []


def ingest_table(table_name, schema_name, trigger_mode='availableNow'):
    is_dim          = schema_name == 'dim'
    source_base     = DIM_PARQUET_BASE     if is_dim else FACT_PARQUET_BASE
    checkpoint_base = DIM_CHECKPOINT_BASE  if is_dim else FACT_CHECKPOINT_BASE

    source_path     = f'{source_base}/{table_name}'
    checkpoint_path = f'{checkpoint_base}/{table_name}/checkpoint'
    schema_location = f'{checkpoint_base}/{table_name}/schema'
    target_table    = f'{CATALOG}.{schema_name}.{table_name}'
    partition_cols  = PARTITIONED_TABLES.get(table_name)

    print(f'  {table_name}  ->  {target_table}')

    reader = (
        spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'parquet')
            .option('cloudFiles.schemaLocation', schema_location)
            .option('cloudFiles.inferColumnTypes', 'false')   # use our explicit schema
            .schema(schema_to_ddl(TABLE_SCHEMAS[table_name]))
            .load(source_path)
    )

    writer = (
        reader.writeStream
            .format('delta')
            .outputMode('append')
            .option('checkpointLocation', checkpoint_path)
            .option('overwriteSchema', 'true')
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    if trigger_mode == 'availableNow':
        # Process all currently available files, then stop automatically
        query = writer.trigger(availableNow=True).toTable(target_table)
        query.awaitTermination()
        print(f'     done  ({query.lastProgress["numInputRows"] if query.lastProgress else "?"} rows in last micro-batch)')
    else:
        # Keep stream alive; picks up new files every 30 s
        query = writer.trigger(processingTime='30 seconds').toTable(target_table)
        active_streams.append((table_name, query))
        print(f'     stream started  (id={query.id})')

    return query


print('ingest_table() ready')

## Tabele wymiarowe

7 tabel – bez partycjonowania.

In [0]:
DIM_TABLES = [
    'dim_locations',
    'dim_employees',
    'dim_customers',
    'dim_vehicles',
    'dim_products',
    'dim_services',
    'dim_suppliers',
]

print('=== Dimension tables ===')
for table in DIM_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'dim', TRIGGER_MODE)
print('Done.')

## Tabele faktowe

13 tabel – 7 partycjonowanych po `rok` / `miesiac` w strukturze katalogów parquet.

In [0]:
FACT_TABLES = [
    'fact_work_orders',           # partycjonowane rok/miesiac
    'fact_work_order_items',
    'fact_sales_transactions',    # partycjonowane rok/miesiac
    'fact_sales_items',
    'fact_invoices',              # partycjonowane rok/miesiac
    'fact_payments',              # partycjonowane rok/miesiac
    'fact_inventory_movements',   # partycjonowane rok/miesiac
    'fact_appointments',          # partycjonowane rok/miesiac
    'fact_purchase_orders',       # partycjonowane rok
    'fact_purchase_order_items',
    'fact_customer_feedback',
    'fact_loyalty_program',
    'fact_employee_schedules',
]

print('=== Fact tables ===')
for table in FACT_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'fact', TRIGGER_MODE)
print('Done.')

## Zatrzymanie wszystkich aktywnych strumieni

Uruchom poniższą komórkę, aby zatrzymać wszystkie aktywne strumienie  
(dotyczy tylko trybu `TRIGGER_MODE = continuous`).

In [0]:
if not active_streams:
    print('No active streams to stop.')
else:
    for table_name, query in active_streams:
        query.stop()
        print(f'Stopped: {table_name}')
    active_streams.clear()
    print('All streams stopped.')

## Walidacja

Liczba wierszy dla każdej wczytanej tabeli.

In [0]:
all_tables = [('dim', t) for t in DIM_TABLES] + [('fact', t) for t in FACT_TABLES]

results = []
for schema_name, table_name in all_tables:
    full_name = f'{CATALOG}.{schema_name}.{table_name}'
    try:
        count = spark.table(full_name).count()
        results.append({'table': full_name, 'row_count': count, 'status': 'OK'})
    except Exception as e:
        results.append({'table': full_name, 'row_count': None, 'status': str(e)[:60]})

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))